# EDA

### Imports + Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Load config
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Plot styling
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries loaded.")

### Load data

In [ ]:
filepath = config["data"]["raw_dir"] + config["data"]["lending_club_file"]
df = pd.read_csv(filepath, low_memory=False)
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head()

### Create binary target variable

In [ ]:
filepath = config["data"]["raw_dir"] + config["data"]["lending_club_file"]
df = pd.read_csv(filepath, low_memory=False)
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head()

### Class imbalance

In [ ]:
counts = df["default"].value_counts()
labels = ["Repaid", "Default"]
colors = ["#4C9BE8", "#E8593C"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, counts.values, color=colors, width=0.5)
ax.set_title("Class distribution: repaid vs default")
ax.set_ylabel("Number of loans")
for i, v in enumerate(counts.values):
    ax.text(i, v + 10000, f"{v:,}\n({v/len(df):.1%})", ha="center", fontsize=11)
plt.tight_layout()
plt.show()

### Default rate by loan grade

In [ ]:
grade_default = df.groupby("grade")["default"].mean().reset_index()
grade_default.columns = ["grade", "default_rate"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(grade_default["grade"], grade_default["default_rate"],
              color=sns.color_palette("RdYlGn_r", len(grade_default)))
ax.set_title("Default rate by loan grade")
ax.set_ylabel("Default rate")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.show()

grade_default

### DTI and income by default status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# DTI distribution
df[df["default"]==0]["dti"].clip(0, 60).plot.hist(
    ax=axes[0], bins=50, alpha=0.6, color="#4C9BE8", label="Repaid")
df[df["default"]==1]["dti"].clip(0, 60).plot.hist(
    ax=axes[0], bins=50, alpha=0.6, color="#E8593C", label="Default")
axes[0].set_title("DTI distribution by default status")
axes[0].set_xlabel("Debt-to-income ratio")
axes[0].legend()

# Annual income distribution (log scale)
df[df["default"]==0]["annual_inc"].clip(0, 300000).plot.hist(
    ax=axes[1], bins=50, alpha=0.6, color="#4C9BE8", label="Repaid")
df[df["default"]==1]["annual_inc"].clip(0, 300000).plot.hist(
    ax=axes[1], bins=50, alpha=0.6, color="#E8593C", label="Default")
axes[1].set_title("Annual income by default status")
axes[1].set_xlabel("Annual income ($)")
axes[1].legend()

plt.tight_layout()
plt.show()

## Key Stats:

In [ ]:
summary = df.groupby("default")[["loan_amnt", "int_rate", "dti", "annual_inc"]].mean()
summary.index = ["Repaid", "Default"]
summary.columns = ["Avg loan amount", "Avg interest rate", "Avg DTI", "Avg annual income"]
summary.round(2)